# 01 — Preparação dos Dados\n\n**Objetivo:** Carregar o corpus da Folha de São Paulo, binarizar labels (mercado vs outros), gerar splits 3-way estratificados (treino/validação/teste), balancear o treino, e persistir tudo em `artifacts/splits/`.\n\n**Dependências:** `economy_classifier.datasets`

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd

from economy_classifier.datasets import build_balanced_training_frame, build_train_val_test_split
from economy_classifier.project import PROJECT_ROOT, SPLITS_DIR, get_git_commit_short, utc_now_iso

In [ ]:
CORPUS_PATH = PROJECT_ROOT / "data" / "news-of-the-site-folhauol" / "articles.csv"
SEED = 42
POSITIVE_LABEL = "mercado"
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

## 1. Carga do corpus

In [ ]:
raw = pd.read_csv(CORPUS_PATH)
print(f"Corpus carregado: {len(raw):,} artigos")
print(f"Colunas: {list(raw.columns)}")
raw["category"].value_counts()

## 2. Binarização e limpeza

In [ ]:
# Binarizar: mercado=1, outros=0
corpus = raw.copy()
corpus[LABEL_COLUMN] = (corpus["category"] == POSITIVE_LABEL).astype(int)

# Remover linhas com texto vazio
corpus = corpus.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)

n_mercado = corpus[LABEL_COLUMN].sum()
n_outros = len(corpus) - n_mercado
print(f"Corpus limpo: {len(corpus):,} artigos")
print(f"  mercado: {n_mercado:,} ({n_mercado/len(corpus)*100:.1f}%)")
print(f"  outros:  {n_outros:,} ({n_outros/len(corpus)*100:.1f}%)")

## 3. Splits 3-way estratificados

In [ ]:
train, val, test = build_train_val_test_split(corpus, label_column=LABEL_COLUMN, seed=SEED)

for name, split in [("Treino", train), ("Validação", val), ("Teste", test)]:
    pct_mercado = split[LABEL_COLUMN].mean() * 100
    print(f"{name:>10}: {len(split):>6,} linhas ({pct_mercado:.1f}% mercado)")

print(f"\n  Total: {len(train)+len(val)+len(test):,} == {len(corpus):,} ✓")

## 4. Balanceamento do treino

In [ ]:
balanced_train = build_balanced_training_frame(train, label_column=LABEL_COLUMN, seed=SEED)
print(f"Treino balanceado: {len(balanced_train):,} linhas")
print(balanced_train[LABEL_COLUMN].value_counts())

## 5. Persistência dos splits

In [ ]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Persistir índices
pd.DataFrame({"index": train.index}).to_csv(SPLITS_DIR / "train_indices.csv", index=False)
pd.DataFrame({"index": val.index}).to_csv(SPLITS_DIR / "val_indices.csv", index=False)
pd.DataFrame({"index": test.index}).to_csv(SPLITS_DIR / "test_indices.csv", index=False)

# Persistir DataFrames completos para uso nos notebooks subsequentes
train.to_parquet(SPLITS_DIR / "train.parquet")
val.to_parquet(SPLITS_DIR / "val.parquet")
test.to_parquet(SPLITS_DIR / "test.parquet")
balanced_train.to_parquet(SPLITS_DIR / "balanced_train.parquet")

# Hash do corpus para rastreabilidade
corpus_bytes = pd.util.hash_pandas_object(corpus).values.tobytes()
corpus_hash = hashlib.sha256(corpus_bytes).hexdigest()

metadata = {
    "seed": SEED,
    "corpus_sha256": corpus_hash,
    "corpus_rows": len(corpus),
    "train_rows": len(train),
    "val_rows": len(val),
    "test_rows": len(test),
    "train_mercado_pct": round(train[LABEL_COLUMN].mean() * 100, 2),
    "val_mercado_pct": round(val[LABEL_COLUMN].mean() * 100, 2),
    "test_mercado_pct": round(test[LABEL_COLUMN].mean() * 100, 2),
    "balanced_train_rows": len(balanced_train),
    "generated_at": utc_now_iso(),
    "git_commit": get_git_commit_short(),
}

(SPLITS_DIR / "split_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
print("Splits persistidos em", SPLITS_DIR)
print(json.dumps(metadata, indent=2))